<a href="https://colab.research.google.com/github/HanzlaAdeel/Centrality-Matrix/blob/master/Leveraging_centrality_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import f_oneway, ttest_ind, ttest_rel, pearsonr, shapiro, levene
from statsmodels.stats.multitest import multipletests
import networkx as nx
import mne

CHANNELS = [
    'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz'
]

CHANNEL_TO_IDX = {ch: idx for idx, ch in enumerate(CHANNELS)}

REGIONS = {
    'Frontal': ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz'],
    'Temporal': ['T3', 'T4', 'T5', 'T6'],
    'Parietal': ['P3', 'P4', 'Pz'],
    'Occipital': ['O1', 'O2']
}

HEMISPHERES = {
    'Left': ['Fp1', 'F3', 'C3', 'P3', 'O1', 'F7', 'T3', 'T5'],
    'Right': ['Fp2', 'F4', 'C4', 'P4', 'O2', 'F8', 'T4', 'T6']
}

TOPOGRAPHY = {
    'Anterior': ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'Fz'],
    'Posterior': ['P3', 'P4', 'O1', 'O2', 'T5', 'T6', 'Pz'],
    'Central': ['C3', 'C4', 'T3', 'T4', 'Cz']
}

## Load Real Dataset



In [ ]:
!ls -R ds004504/

ds004504/:
CHANGES			  sub-009  sub-023  sub-037  sub-051  sub-065  sub-079
dataset_description.json  sub-010  sub-024  sub-038  sub-052  sub-066  sub-080
derivatives		  sub-011  sub-025  sub-039  sub-053  sub-067  sub-081
participants.json	  sub-012  sub-026  sub-040  sub-054  sub-068  sub-082
participants.tsv	  sub-013  sub-027  sub-041  sub-055  sub-069  sub-083
README			  sub-014  sub-028  sub-042  sub-056  sub-070  sub-084
sub-001			  sub-015  sub-029  sub-043  sub-057  sub-071  sub-085
sub-002			  sub-016  sub-030  sub-044  sub-058  sub-072  sub-086
sub-003			  sub-017  sub-031  sub-045  sub-059  sub-073  sub-087
sub-004			  sub-018  sub-032  sub-046  sub-060  sub-074  sub-088
sub-005			  sub-019  sub-033  sub-047  sub-061  sub-075
sub-006			  sub-020  sub-034  sub-048  sub-062  sub-076
sub-007			  sub-021  sub-035  sub-049  sub-063  sub-077
sub-008			  sub-022  sub-036  sub-050  sub-064  sub-078

ds004504/derivatives:
sub-001  sub-011  sub-021  sub-031  sub-041  sub-051  sub-061

### Load Participant Data



In [ ]:
cohort_df = pd.read_csv('ds004504/participants.tsv', sep='\t')
cohort_df.rename(columns={'participant_id': 'ID', 'sex': 'Gender', 'age': 'Age', 'group': 'Group', 'mmse': 'MMSE'}, inplace=True)

# The actual group names in participants.tsv are 'AD', 'HC', 'FTD'. We found 'AD' is represented as 'A'.
# So, map 'A' to 'AD', 'HC' to 'C', and 'FTD' to 'F'.
cohort_df['Group'] = cohort_df['Group'].replace({'A': 'AD', 'HC': 'C', 'FTD': 'F'})

print(cohort_df.head())
print("Unique groups in cohort_df after mapping:", cohort_df['Group'].unique())

        ID Gender  Age Group  MMSE
0  sub-001      F   57    AD    16
1  sub-002      F   78    AD    22
2  sub-003      M   70    AD    14
3  sub-004      F   67    AD    20
4  sub-005      M   70    AD    22
Unique groups in cohort_df after mapping: ['AD' 'C' 'F']


###  MNE-Python for EEG Data Loading



In [ ]:
!pip install mne

###  EEG Loading Function



In [ ]:
def compute_pli(eeg_data: np.ndarray) -> np.ndarray:
    n_channels, n_samples = eeg_data.shape
    analytic_signal = hilbert(eeg_data, axis=-1)
    phase_data = np.angle(analytic_signal)

    pli_matrix = np.zeros((n_channels, n_channels))
    for i in range(n_channels):
        for j in range(i + 1, n_channels):
            delta_phase = phase_data[i] - phase_data[j]
            pli = np.abs(np.mean(np.sign(np.sin(delta_phase))))
            pli_matrix[i, j] = pli
            pli_matrix[j, i] = pli

    np.fill_diagonal(pli_matrix, 0.0)
    return pli_matrix

In [ ]:
def compute_centralities(adj_matrix: np.ndarray) -> dict:
    n_nodes = adj_matrix.shape[0]
    G = nx.from_numpy_array(adj_matrix)
    degrees = np.array([val for _, val in G.degree(weight='weight')])

    G_dist = nx.Graph()
    for u, v, data in G.edges(data=True):
        w = data['weight']
        if w > 0:
            G_dist.add_edge(u, v, weight=1.0 / w)

    lev_centrality = np.zeros(n_nodes)
    for v in range(n_nodes):
        neighbors = list(G.neighbors(v))
        if len(neighbors) == 0 or degrees[v] == 0:
            continue
        sum_terms = sum((degrees[v] - degrees[vi]) / (degrees[v] + degrees[vi])
                        for vi in neighbors if (degrees[v] + degrees[vi]) > 0)
        lev_centrality[v] = sum_terms / len(neighbors)

    try:
        eig_cen = nx.eigenvector_centrality(G, weight='weight', max_iter=1000, tol=1e-6)
    except nx.PowerIterationFailedConvergence:
        eig_cen = {i: 1.0 / np.sqrt(n_nodes) for i in range(n_nodes)}

    C_E = np.array([eig_cen[i] for i in range(n_nodes)])

    W_factor = np.zeros(n_nodes)
    for v in range(n_nodes):
        neighbors = list(G.neighbors(v))
        total_v = C_E[v] + sum(C_E[vi] * adj_matrix[v, vi] for vi in neighbors)
        W_factor[v] = C_E[v] / (total_v if total_v != 0 else 1.0)

    weighted_lev = np.zeros(n_nodes)
    for v in range(n_nodes):
        neighbors = list(G.neighbors(v))
        if len(neighbors) == 0 or degrees[v] == 0:
            continue
        sum_terms = 0.0
        for vi in neighbors:
            w_deg_v = W_factor[v] * degrees[v]
            w_deg_vi = W_factor[vi] * degrees[vi]
            denom = w_deg_v + w_deg_vi
            if denom > 0:
                sum_terms += (w_deg_v - w_deg_vi) / denom
        weighted_lev[v] = sum_terms / len(neighbors)

    bet_cen_dict = nx.betweenness_centrality(G_dist, weight='weight', normalized=True)
    bet_cen = np.array([bet_cen_dict[i] for i in range(n_nodes)])

    clos_cen = np.zeros(n_nodes)
    for v in range(n_nodes):
        try:
            lengths = nx.single_source_dijkstra_path_length(G_dist, v, weight='weight')
            total_dist = sum(lengths[u] for u in range(n_nodes) if u != v and u in lengths)
            if total_dist > 0:
                clos_cen[v] = (n_nodes - 1.0) / total_dist
        except Exception:
            clos_cen[v] = 0.0

    return {
        'Leverage': lev_centrality,
        'Weighted_Leverage': weighted_lev,
        'Betweenness': bet_cen,
        'Closeness': clos_cen
    }

In [ ]:
def aggregate_metrics(centrality_array: np.ndarray, channel_groups: dict) -> dict:
    scores = {}
    for region_name, ch_list in channel_groups.items():
        indices = [CHANNEL_TO_IDX[ch] for ch in ch_list]
        scores[region_name] = np.mean(centrality_array[indices])
    return scores

In [ ]:
def run_statistical_pipeline(df_scores: pd.DataFrame, metric_col: str, group_col='Group'):
    groups = df_scores[group_col].unique()
    grouped_data = [df_scores[df_scores[group_col] == g][metric_col].values for g in groups]

    f_stat, p_val = f_oneway(*grouped_data)

    pairwise_comps = [('AD', 'C'), ('C', 'F'), ('F', 'AD')]
    p_vals = []
    t_stats = []
    comparisons = []

    for g1, g2 in pairwise_comps:
        d1 = df_scores[df_scores[group_col] == g1][metric_col].values
        d2 = df_scores[df_scores[group_col] == g2][metric_col].values
        t_stat, p = ttest_ind(d1, d2)
        comparisons.append(f"{g1} vs {g2}")
        t_stats.append(t_stat)
        p_vals.append(p)

    _, fdr_p_vals, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')

    res_df = pd.DataFrame({
        'Comparison': comparisons,
        'T-Statistic': t_stats,
        'p-value': p_vals,
        'FDR p-value': fdr_p_vals,
        'Significant (FDR)': [p < 0.05 for p in fdr_p_vals]
    })

    return f_stat, p_val, res_df

In [ ]:
def generate_study_cohort() -> pd.DataFrame:
    data = [
        ('Sub-01', 'F', 57, 'AD', 16), ('Sub-02', 'F', 78, 'AD', 22),
        ('Sub-03', 'M', 70, 'AD', 14), ('Sub-04', 'F', 67, 'AD', 20),
        ('Sub-05', 'M', 70, 'AD', 22), ('Sub-06', 'F', 61, 'AD', 14),
        ('Sub-07', 'F', 79, 'AD', 20), ('Sub-08', 'M', 62, 'AD', 16),
        ('Sub-09', 'F', 77, 'AD', 23), ('Sub-10', 'M', 69, 'AD', 20),
        ('Sub-11', 'M', 71, 'AD', 22), ('Sub-12', 'M', 63, 'AD', 18),
        ('Sub-13', 'F', 64, 'AD', 20), ('Sub-14', 'M', 77, 'AD', 14),
        ('Sub-15', 'M', 61, 'AD', 18),
        ('Sub-16', 'M', 57, 'C', 30), ('Sub-17', 'M', 62, 'C', 30),
        ('Sub-18', 'M', 70, 'C', 30), ('Sub-19', 'M', 61, 'C', 30),
        ('Sub-20', 'F', 77, 'C', 30), ('Sub-21', 'M', 74, 'C', 30),
        ('Sub-22', 'M', 72, 'C', 30), ('Sub-23', 'F', 64, 'C', 30),
        ('Sub-24', 'F', 70, 'C', 30), ('Sub-25', 'M', 63, 'C', 30),
        ('Sub-26', 'F', 70, 'C', 30), ('Sub-27', 'M', 65, 'C', 30),
        ('Sub-28', 'F', 62, 'C', 30), ('Sub-29', 'M', 68, 'C', 30),
        ('Sub-30', 'F', 75, 'C', 30),
        ('Sub-31', 'M', 73, 'F', 20), ('Sub-32', 'M', 66, 'F', 24),
        ('Sub-33', 'M', 78, 'F', 25), ('Sub-34', 'M', 70, 'F', 22),
        ('Sub-35', 'F', 67, 'F', 22), ('Sub-36', 'M', 62, 'F', 20),
        ('Sub-37', 'M', 65, 'F', 18), ('Sub-38', 'F', 57, 'F', 22),
        ('Sub-39', 'F', 53, 'F', 20), ('Sub-40', 'F', 71, 'F', 22),
        ('Sub-41', 'M', 44, 'F', 24), ('Sub-42', 'M', 61, 'F', 22),
        ('Sub-43', 'M', 62, 'F', 22), ('Sub-44', 'F', 60, 'F', 18),
        ('Sub-45', 'F', 71, 'F', 20)
    ]
    return pd.DataFrame(data, columns=['ID', 'Gender', 'Age', 'Group', 'MMSE'])

In [ ]:
fs = 250
duration = 10
n_samples = fs * duration

def load_eeg_data(subject_id: str) -> np.ndarray:
    n_channels = len(CHANNELS)
    eeg_data = np.random.randn(n_channels, n_samples)

    nyquist = 0.5 * fs
    low = 8.0 / nyquist
    high = 12.0 / nyquist
    b, a = butter(5, [low, high], btype='band')
    eeg_data = filtfilt(b, a, eeg_data, axis=-1)

    print(f"Generated synthetic EEG data for {subject_id} with shape {eeg_data.shape}")
    return eeg_data

n_subjects = len(cohort_df)

participant_results = []

for idx, row in cohort_df.iterrows():
    subject_id = row['ID']
    eeg_data = load_eeg_data(subject_id)

    if eeg_data is None:
        print(f"Skipping {subject_id} due to EEG loading error.")
        continue

    if eeg_data.shape[0] != len(CHANNELS):
        print(f"Warning: EEG data for {subject_id} has {eeg_data.shape[0]} channels, expected {len(CHANNELS)}. Skipping.")
        continue

    pli_mat = compute_pli(eeg_data)
    cens = compute_centralities(pli_mat)

    row_entry = dict(row)
    for metric_name, node_vals in cens.items():
        anat_scores = aggregate_metrics(node_vals, REGIONS)
        for r_name, val in anat_scores.items():
            row_entry[f"{metric_name}_{r_name}"] = val

        hemi_scores = aggregate_metrics(node_vals, HEMISPHERES)
        for h_name, val in hemi_scores.items():
            row_entry[f"{metric_name}_{h_name}"] = val

        topo_scores = aggregate_metrics(node_vals, TOPOGRAPHY)
        for t_name, val in topo_scores.items():
            row_entry[f"{metric_name}_{t_name}"] = val

    participant_results.append(row_entry)

df_results = pd.DataFrame(participant_results)

df_results['MMSE'] = pd.to_numeric(df_results['MMSE'], errors='coerce')

valid_groups = ['AD', 'C', 'F']
df_results_filtered_groups = df_results[df_results['Group'].isin(valid_groups)].copy()

for region in ['Frontal', 'Temporal', 'Parietal', 'Occipital']:
    col = f"Closeness_{region}"
    if len(df_results_filtered_groups['Group'].unique()) > 1 and len(df_results_filtered_groups) > len(df_results_filtered_groups['Group'].unique()):
        f_stat, p_val, pairwise_df = run_statistical_pipeline(df_results_filtered_groups, col, group_col='Group')
        print(f"\n[Region: {region}] One-Way ANOVA: F = {f_stat:.4f}, p = {p_val:.4f}")
        print(pairwise_df[['Comparison', 'T-Statistic', 'p-value', 'FDR p-value', 'Significant (FDR)']].to_string(index=False))
    else:
        print(f"\n[Region: {region}] Not enough data or groups for ANOVA.")

for group in ['AD', 'F', 'C']:
    sub_df = df_results_filtered_groups[df_results_filtered_groups['Group'] == group].dropna(subset=['MMSE'])

    if len(sub_df) > 1 and sub_df['Leverage_Temporal'].nunique() > 1 and sub_df['Age'].nunique() > 1:
        r_age, p_age = pearsonr(sub_df['Leverage_Temporal'], sub_df['Age'])
    else:
        r_age, p_age = np.nan, np.nan

    if len(sub_df) > 1 and sub_df['Betweenness_Central'].nunique() > 1 and sub_df['MMSE'].nunique() > 1:
        r_mmse, p_mmse = pearsonr(sub_df['Betweenness_Central'], sub_df['MMSE'])
    else:
        r_mmse, p_mmse = np.nan, np.nan

    print(f"\nGroup {group:3s} | Leverage (Temporal) vs Age:  r = {r_age:+.3f} (p = {p_age:.4f})")
    print(f"Group {group:3s} | Betweenness (Central) vs MMSE: r = {r_mmse:+.3f} (p = {p_mmse:.4f})")

Generated synthetic EEG data for sub-001 with shape (19, 2500)
Generated synthetic EEG data for sub-002 with shape (19, 2500)
Generated synthetic EEG data for sub-003 with shape (19, 2500)
Generated synthetic EEG data for sub-004 with shape (19, 2500)
Generated synthetic EEG data for sub-005 with shape (19, 2500)
Generated synthetic EEG data for sub-006 with shape (19, 2500)
Generated synthetic EEG data for sub-007 with shape (19, 2500)
Generated synthetic EEG data for sub-008 with shape (19, 2500)
Generated synthetic EEG data for sub-009 with shape (19, 2500)
Generated synthetic EEG data for sub-010 with shape (19, 2500)
Generated synthetic EEG data for sub-011 with shape (19, 2500)
Generated synthetic EEG data for sub-012 with shape (19, 2500)
Generated synthetic EEG data for sub-013 with shape (19, 2500)
Generated synthetic EEG data for sub-014 with shape (19, 2500)
Generated synthetic EEG data for sub-015 with shape (19, 2500)
Generated synthetic EEG data for sub-016 with shape (19